# Medical regulation assistant: routed RAG over the French *Guide de régulation médicale*

This notebook rebuilds the chatbot we developed during our PSC with the Agence du Numérique en Santé (ANS).
It helps medical regulation assistants (ARM) and regulating physicians find the relevant procedure in the
[Guide d'aide à la régulation médicale](https://www.guide-regulation-medicale.fr) while handling a call.

**Pipeline**

```
question ──> 1. page classifier (LLM) ──> 2. retrieval restricted to that page (FAISS) ──> 3. grounded answer (LLM) + link to the page
```

1. The LLM reads the question and picks the most relevant page among the ~190 pages of the guide.
2. The passages closest to the question are retrieved, **only within that page**.
3. The LLM answers from those passages and the link to the source page is shown.

All models come from the Hugging Face Hub. With a GPU they run locally, otherwise they are called
through the Hugging Face Inference API (a free `HF_TOKEN` is enough).

> This is a research prototype, not a medical device. Answers must always be checked against the guide.

## 1. Configuration

In [12]:
import os
import json
import difflib
import getpass
from pathlib import Path

os.environ.setdefault("USER_AGENT", "medical-regulation-rag/1.0")

# Models (any compatible model from the Hugging Face Hub can be used)
LLM_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
EMBEDDING_MODEL = "BAAI/bge-m3"

# Retrieval parameters (same as in the original project)
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 6
EMBED_BATCH_SIZE = 32  # chunks per embedding request (HuggingFace does not respond if the batch size is too important)

DATA_DIR = Path("data")
CACHE_DIR = DATA_DIR / "cache"
PAGES_CACHE = CACHE_DIR / "pages.json"
INDEX_DIR = CACHE_DIR / f"faiss_{EMBEDDING_MODEL.replace('/', '_')}"

try:
    import torch
    USE_GPU = torch.cuda.is_available()
except ImportError:
    USE_GPU = False

print(f"Mode: {'local GPU' if USE_GPU else 'Hugging Face Inference API'}")

Mode: Hugging Face Inference API


## 2. Models

- **GPU available**: the LLM and the embedding model are downloaded from the Hub and run locally with `transformers`.
- **No GPU**: the same models are called through the Hugging Face Inference API.
You can force the use of the API even if you have a GPU by setting `USE_API = True`.

In [13]:
from langchain_huggingface import ChatHuggingFace

if not USE_GPU and not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token: ")

if USE_GPU:
    from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline

    llm_backend = HuggingFacePipeline.from_model_id(
        model_id=LLM_MODEL,
        task="text-generation",
        device_map="auto",
        model_kwargs={"torch_dtype": "auto"},
        pipeline_kwargs={"max_new_tokens": 512, "do_sample": False, "return_full_text": False},
    )
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={"device": "cuda"},
        encode_kwargs={"normalize_embeddings": True},
    )
else:
    from langchain_huggingface import HuggingFaceEndpoint, HuggingFaceEndpointEmbeddings

    llm_backend = HuggingFaceEndpoint(
        repo_id=LLM_MODEL,
        task="text-generation",
        max_new_tokens=512,
        temperature=0.1,
    )
    embeddings = HuggingFaceEndpointEmbeddings(model=EMBEDDING_MODEL)

llm = ChatHuggingFace(llm=llm_backend)

## 3. Loading the guide

The list of pages (category, title, URL) is stored in `data/guide_pages.json`.
Each page is scraped once, keeping only the procedure itself (`div.fiche_content`, without the site navigation),
and cached in `data/cache/pages.json`.

In [14]:
from bs4 import SoupStrainer
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document

guide_pages = json.loads((DATA_DIR / "guide_pages.json").read_text(encoding="utf-8"))
page_to_category = {page: category for category, pages in guide_pages.items() for page in pages}
page_to_url = {page: url for pages in guide_pages.values() for page, url in pages.items()}
print(f"{len(guide_pages)} categories, {len(page_to_url)} pages")


def scrape_guide():
    pages = []
    for category, category_pages in guide_pages.items():
        for page, url in category_pages.items():
            loader = WebBaseLoader(
                url,
                bs_kwargs={"parse_only": SoupStrainer(class_="fiche_content")},
                bs_get_text_kwargs={"separator": "\n", "strip": True},
            )
            try:
                text = loader.load()[0].page_content
            except Exception as e:
                print(f"Could not load {category} / {page}: {e}")
                continue
            pages.append({"category": category, "page": page, "url": url, "text": text})
    return pages


if PAGES_CACHE.exists():
    pages = json.loads(PAGES_CACHE.read_text(encoding="utf-8"))
else:
    pages = scrape_guide()
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    PAGES_CACHE.write_text(json.dumps(pages, ensure_ascii=False), encoding="utf-8")

documents = [
    Document(
        page_content=f"{p['page']}\n{p['text']}",
        metadata={"category": p["category"], "page": p["page"], "url": p["url"]},
    )
    for p in pages
    if p["text"]
]
print(f"{len(documents)} pages loaded")

27 categories, 193 pages
193 pages loaded


## 4. Vector index

All pages are split into overlapping chunks and stored in a single FAISS index.
Each chunk keeps its page title as metadata, so retrieval can later be restricted to one page.
Chunks are embedded in small batches (a single request with every chunk times out on the Inference API).
The index is saved to disk and reloaded on the next run.

In [15]:
import time

from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter


def embed_batch(vectorstore, batch, retries=3):
    for attempt in range(retries):
        try:
            if vectorstore is None:
                return FAISS.from_documents(batch, embeddings)
            vectorstore.add_documents(batch)
            return vectorstore
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f"\nRetrying after error: {e}")
            time.sleep(10)


if INDEX_DIR.exists():
    vectorstore = FAISS.load_local(INDEX_DIR, embeddings, allow_dangerous_deserialization=True)
else:
    splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    chunks = splitter.split_documents(documents)
    vectorstore = None
    for start in range(0, len(chunks), EMBED_BATCH_SIZE):
        vectorstore = embed_batch(vectorstore, chunks[start:start + EMBED_BATCH_SIZE])
        print(f"\r{min(start + EMBED_BATCH_SIZE, len(chunks))}/{len(chunks)} chunks embedded", end="")
    print()
    vectorstore.save_local(INDEX_DIR)

print(f"{vectorstore.index.ntotal} chunks indexed")

1166 chunks indexed


## 5. Page classifier

The LLM receives the table of contents of the guide and returns the title of the most relevant page.
Its output is then snapped to the closest existing title, so a small formatting deviation cannot break the pipeline.

In [16]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

catalog = "\n".join(f"- {category} : " + " | ".join(pages) for category, pages in guide_pages.items())

classification_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Tu aides un assistant de régulation médicale du SAMU à trouver la bonne fiche "
        "du guide de régulation médicale. Voici les fiches du guide, regroupées par catégorie :\n"
        "{catalog}\n\n"
        "Réponds uniquement par le titre exact de la fiche la plus pertinente pour la question, "
        "sans aucun autre texte.",
    ),
    ("human", "{question}"),
]).partial(catalog=catalog)

classification_chain = classification_prompt | llm | StrOutputParser()


def classify(question):
    raw = classification_chain.invoke({"question": question}).strip()
    return difflib.get_close_matches(raw, list(page_to_url), n=1, cutoff=0.0)[0]

## 6. Grounded answer

Retrieval is restricted to the selected page (`filter={"page": ...}`), then the LLM answers
from the retrieved passages only. The prompt insists on staying factual and on saying so when the answer is not in the guide.

In [17]:
from IPython.display import Markdown, display

answer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Tu es un assistant qui aide les assistants de régulation médicale et les médecins régulateurs "
        "à consulter le guide de régulation médicale. Utilise uniquement les extraits suivants de la fiche "
        "« {page} » pour répondre à la question, de la manière la plus précise et exhaustive possible. "
        "N'invente rien : si les extraits ne permettent pas de répondre, dis-le.\n\n{context}",
    ),
    ("human", "{question}"),
])

answer_chain = answer_prompt | llm | StrOutputParser()


def ask(question, show=True):
    page = classify(question)
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": TOP_K, "filter": {"page": page}, "fetch_k": vectorstore.index.ntotal}
    )
    passages = retriever.invoke(question)
    context = "\n\n".join(p.page_content for p in passages)
    answer = answer_chain.invoke({"page": page, "context": context, "question": question})

    result = {
        "category": page_to_category[page],
        "page": page,
        "url": page_to_url[page],
        "answer": answer,
        "passages": passages,
    }
    if show:
        display(Markdown(
            f"**Question :** {question}\n\n"
            f"**Fiche :** {result['category']} / [{page}]({result['url']})\n\n"
            f"{answer}"
        ))
    return result

## 7. Examples

In [19]:
_ = ask("Je saigne de la main")

**Question :** Je saigne de la main

**Fiche :** TRAUMATOLOGIE / [Traumatisme isolé d’un doigt ou d’une main](https://www.guide-regulation-medicale.fr/index.php?module=Fiche&action=FrontDetailView&record=965)

En fonction des extraits fournis, voici les conseils médicaux à suivre :

* Allonger le blessé la main atteinte surélevée si vous avez une mauvaise tolérance.
* Si vous saignez activement, appliquez un point de compression si possible avec un linge propre.

Il est important de noter que vous devez attendre l'arrivée des secours ou de la régulation médicale pour recevoir un examen médical approprié. Les traumatismes de la main peuvent nécessiter une exploration chirurgicale, et il est préférable de laisser les professionnels du secours et les médecins régulateurs prendre les décisions concernant votre traitement.

In [20]:
_ = ask("Le patient a une douleur dans la poitrine depuis 30 minutes, quel niveau de priorité ?")

**Question :** Le patient a une douleur dans la poitrine depuis 30 minutes, quel niveau de priorité ?

**Fiche :** MOTIFS DE RECOURS / [Degrés d’urgence en régulation médicale : les P, R et B](https://www.guide-regulation-medicale.fr/index.php?module=Fiche&action=FrontDetailView&record=861)

D'après les extraits fournis, cette situation correspondrait à un niveau P3 : régulation médicale reportée ou programmée et traitée par un rappel. En effet, la douleur dans la poitrine est une situation non urgente qui nécessite un transfert inter-hospitalier programmé, et le délai prévisible de rappel sera donné à l'appelant à titre indicatif.

In [21]:
_ = ask("Un enfant de 2 ans convulse avec de la fièvre, que conseiller en attendant les secours ?")

**Question :** Un enfant de 2 ans convulse avec de la fièvre, que conseiller en attendant les secours ?

**Fiche :** PEDIATRIE / [Convulsions de l'enfant](https://www.guide-regulation-medicale.fr/index.php?module=Fiche&action=FrontDetailView&record=280)

D'après les extraits fournis, en attendant les secours, il est conseillé de :

* Ne pas tenter d'empêcher les mouvements, ne pas introduire d'objet dans la bouche, écarter objets et meubles pouvant blesser l'enfant.
* Rassurer et expliquer le déroulement normal d'une crise et la lenteur du réveil, expliquer la mise en position latérale de sécurité.
* En cas de convulsion hyperthermique, déshabiller l'enfant, mettre en position latérale de sécurité et dégager la bouche.
* Conseiller aux parents qui en disposent l'administration d'anticonvulsivants en précisant la dose et la surveillance (l'appel est souvent pour valider cette administration).

Il est important de noter que si la convulsion persiste, il est conseillé d'engager immédiatement un moyen secouriste équipé d'oxygène.

In [18]:
_ = ask("AVC")

**Question :** AVC

**Fiche :** CARDIO-VASCULAIRE / [Arrêt cardiaque de l’adulte](https://www.guide-regulation-medicale.fr/index.php?module=Fiche&action=FrontDetailView&record=104)

Je vais répondre à votre question en utilisant les extraits fournis de la fiche "Arrêt cardiaque de l'adulte".

Lorsqu'un AVC est suspecté, le médecin régulateur devrait :

* Confirmer l'arrêt cardiaque : "Est-ce qu'il parle ? Est-ce qu'il bouge ?" (élément d'analyse et critère de gravité)
* Se méfier des gasps et des convulsions qui masquent les signes habituels d'arrêt cardiaque (élément d'analyse et critère de gravité)
* Rechercher les circonstances : malaise devant témoin, plaintes précédent l'arrêt cardiaque (élément d'analyse et critère de gravité)
* Demander si une réanimation cardiopulmonaire est débutée (élément d'analyse et critère de gravité)
* Faire débuter et accompagner la réanimation cardiaque jusqu'à l'arrivée des secours (conseil médical)
* Faire preuve d'empathie, rendre le témoin efficace sans le culpabiliser (conseil médical)
* Faire appeler de l'aide dans l'environnement (conseil médical)
* Faire installer le patient sur le dos, si possible sur un plan dur (conseil médical)

Il est important de noter que l'arrêt cardiaque et l'AVC sont des situations médicales distinctes qui nécessitent des interventions différentes. Lorsqu'un AVC est suspecté, il est essentiel de prendre en charge le patient de manière appropriée et de le faire transporter vers un centre de réanimation ou un hôpital pour une prise en charge spécialisée.